In [6]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, 
                             confusion_matrix, roc_auc_score)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.ae import AnomalyDetector

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "AnomalyAE"
TARGET_FAR = 0.05   # 5% false alarm rate on healthy samples


device: NVIDIA A30


In [7]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def stratified_balanced_test(X_te, y_te, n_f0_target=None, seed=0):
    """
    Build a stratified test set where total faults = total F0 samples.
    Each fault class contributes equally.
    """
    rng = np.random.RandomState(seed)
    y_np = y_te.numpy() if hasattr(y_te, "numpy") else np.asarray(y_te)
    n_classes = int(y_np.max()) + 1

    # F0 samples — keep all, or downsample if n_f0_target is smaller
    f0_idx = np.where(y_np == 0)[0]
    if n_f0_target is not None and n_f0_target < len(f0_idx):
        f0_idx = rng.choice(f0_idx, size=n_f0_target, replace=False)
    n_f0 = len(f0_idx)

    # Fault samples — n_f0 // (n_classes - 1) per fault class
    n_per_fault = n_f0 // (n_classes - 1)
    fault_idx_list = []
    for c in range(1, n_classes):
        c_idx = np.where(y_np == c)[0]
        if len(c_idx) >= n_per_fault:
            chosen = rng.choice(c_idx, size=n_per_fault, replace=False)
        else:
            chosen = c_idx
        fault_idx_list.append(chosen)
    fault_idx = np.concatenate(fault_idx_list)

    all_idx = np.concatenate([f0_idx, fault_idx])
    rng.shuffle(all_idx)

    X_balanced = X_te[all_idx]
    y_balanced = y_te[all_idx] if hasattr(y_te, "numpy") else y_np[all_idx]
    y_balanced_binary = (y_np[all_idx] != 0).astype(int)

    return X_balanced, y_balanced, y_balanced_binary


def _eval_block(detector, X, y_true_binary, y_true_multi):
    """Compute the standard metric bundle on any (X, y) pair.
    Returns a dict; keeps the calling code DRY across imbalanced vs balanced."""
    y_pred  = detector.predict(X)
    y_score = detector.score(X)

    acc = accuracy_score(y_true_binary, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true_binary, y_pred, average="binary", zero_division=0)
    auc = roc_auc_score(y_true_binary, y_score)
    cm  = confusion_matrix(y_true_binary, y_pred, labels=[0, 1])

    per_class_detection = {}
    y_multi = y_true_multi.numpy() if hasattr(y_true_multi, "numpy") \
              else np.asarray(y_true_multi)
    n_classes = int(y_multi.max()) + 1
    for c in range(n_classes):
        mask = (y_multi == c)
        if mask.sum() > 0:
            per_class_detection[c] = float(y_pred[mask].mean())

    return {
        "accuracy": acc, "precision": p, "recall": r, "f1": f1,
        "auc": auc, "confusion": cm,
        "per_class_detection": per_class_detection,
        "n_f0":    int((y_true_binary == 0).sum()),
        "n_fault": int((y_true_binary == 1).sum()),
    }


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4,
                 target_far=TARGET_FAR, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    # F0-only splits for unsupervised AE training and threshold calibration
    X_tr_f0 = X_tr[y_tr == 0]
    X_va_f0 = X_va[y_va == 0]

    # Build the AE detector
    detector = AnomalyDetector(
        in_dim=13, hidden_dims=(64, 32), latent_dim=16,
        device=device, seed=seed,
        score_mode="mse",   # or "mahalanobis" or "combined"
    )
    n_params, params_by_type = count_parameters(detector.model)
    reset_gpu_peak_memory(device)

    # Header
    y_te_binary_imbal = (y_te.numpy() != 0).astype(int)
    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  full train: {len(X_tr)}  F0-only: {len(X_tr_f0)}  "
              f"F0-val: {len(X_va_f0)}  test: {len(X_te)}")
        print(f"  imbalanced test: F0={int((y_te_binary_imbal == 0).sum())}  "
              f"fault={int((y_te_binary_imbal == 1).sum())}")
        print(f"  AE params: {n_params:,}  breakdown: {params_by_type}")

    # --- training & calibration ---
    with Timer(device) as train_timer:
        detector.fit(X_tr_f0, X_f0_val=X_va_f0,
                     epochs=epochs, lr=lr, weight_decay=weight_decay,
                     batch_size=50, seed=seed)
        thresh = detector.calibrate(X_va_f0, target_far=target_far)

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    if verbose:
        print(f"  threshold (FAR={target_far}): {thresh:.6f}")

    # --- inference timing on the original test set ---
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(detector.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # --- imbalanced eval (original test set, used in cross-model comparison) ---
    imbal = _eval_block(detector, X_te, y_te_binary_imbal, y_te)

    # --- balanced eval (stratified test set, AE-specific honest F1) ---
    X_te_bal, y_te_bal_multi, y_te_bal_binary = stratified_balanced_test(
        X_te, y_te, seed=seed)
    bal = _eval_block(detector, X_te_bal, y_te_bal_binary, y_te_bal_multi)

    if verbose:
        print(f"  IMBALANCED (F0={imbal['n_f0']}, fault={imbal['n_fault']}): "
              f"acc={imbal['accuracy']:.4f}  P={imbal['precision']:.4f}  "
              f"R={imbal['recall']:.4f}  F1={imbal['f1']:.4f}  "
              f"AUC={imbal['auc']:.4f}")
        print(f"  BALANCED   (F0={bal['n_f0']}, fault={bal['n_fault']}): "
              f"acc={bal['accuracy']:.4f}  P={bal['precision']:.4f}  "
              f"R={bal['recall']:.4f}  F1={bal['f1']:.4f}  "
              f"AUC={bal['auc']:.4f}")
        print(f"  per-class flag rate (imbalanced): " +
              " ".join(f"F{c}={v:.2f}"
                       for c, v in imbal["per_class_detection"].items()))
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":            scenario_idx,
        "model":               MODEL_NAME,
        "n_train_total":       len(X_tr),
        "n_train_f0":          len(X_tr_f0),
        # imbalanced (cross-model comparison) -- preserves original column names
        "accuracy_binary":     imbal["accuracy"],
        "precision_binary":    imbal["precision"],
        "recall_binary":       imbal["recall"],
        "f1_binary":           imbal["f1"],
        "auc":                 imbal["auc"],
        "confusion_binary":    imbal["confusion"],
        "per_class_detection": imbal["per_class_detection"],
        # balanced (AE-honest) -- new columns
        "n_test_balanced":     imbal["n_f0"] + bal["n_fault"],
        "accuracy_bal":        bal["accuracy"],
        "precision_bal":       bal["precision"],
        "recall_bal":          bal["recall"],
        "f1_bal":              bal["f1"],
        "auc_bal":             bal["auc"],
        "confusion_bal":       bal["confusion"],
        "per_class_detection_bal": bal["per_class_detection"],
        # housekeeping
        "threshold":           thresh,
        "n_params":            n_params,
        "train_sec":           round(train_timer.elapsed, 2),
        "inf_ms_per_sample":   round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb":         round(peak_mem_mb, 1),
    }

In [8]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-3, seed=0)
    results.append(r)



=== Scenario 1 (AnomalyAE) ===
  full train: 7858  F0-only: 858  F0-val: 200  test: 1600
  imbalanced test: F0=200  fault=1400
  AE params: 7,005  breakdown: {'encoder': 3504, 'decoder': 3501}
  threshold (FAR=0.05): 0.003557
  IMBALANCED (F0=200, fault=1400): acc=0.7006  P=0.9904  R=0.6643  F1=0.7952  AUC=0.9494
  BALANCED   (F0=200, fault=196): acc=0.7854  P=0.9302  R=0.6122  F1=0.7385  AUC=0.9428
  per-class flag rate (imbalanced): F0=0.04 F1=0.52 F2=0.97 F3=0.14 F4=1.00 F5=0.66 F6=0.63 F7=0.72
  COMPUTE: train=3.5s  inf=0.000ms/sample  peak_mem=17.8MB

=== Scenario 2 (AnomalyAE) ===
  full train: 3939  F0-only: 439  F0-val: 200  test: 1600
  imbalanced test: F0=200  fault=1400
  AE params: 7,005  breakdown: {'encoder': 3504, 'decoder': 3501}
  threshold (FAR=0.05): 0.004710
  IMBALANCED (F0=200, fault=1400): acc=0.6800  P=0.9784  R=0.6486  F1=0.7801  AUC=0.8931
  BALANCED   (F0=200, fault=196): acc=0.7803  P=0.8658  R=0.6582  F1=0.7478  AUC=0.8984
  per-class flag rate (imbalanced

In [9]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train_total", "n_train_f0",
                       "accuracy_binary", "precision_binary", "recall_binary",
                       "f1_binary", "auc", "n_params", "train_sec",
                       "inf_ms_per_sample", "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy_binary", "precision_binary", "recall_binary",
              "f1_binary", "auc"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios ===")
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)



=== AnomalyAE summary across scenarios ===
 scenario     model  n_train_total  n_train_f0  accuracy_binary  precision_binary  recall_binary  f1_binary    auc  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 AnomalyAE           7858         858           0.7006            0.9904         0.6643     0.7952 0.9494      7005       3.54             0.0003         17.8
        2 AnomalyAE           3939         439           0.6800            0.9784         0.6486     0.7801 0.8931      7005       1.73             0.0003         17.6
        3 AnomalyAE            786          86           0.4094            0.9691         0.3357     0.4987 0.7335      7005       0.39             0.0003         17.4
        4 AnomalyAE            391          41           0.2769            0.9451         0.1843     0.3084 0.6215      7005       0.20             0.0003         17.4
        5 AnomalyAE            238          28           0.2625            0.9508         0.1657     0.2822 0.6023  

In [10]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_f0={r['n_train_f0']}, acc_bin={r['accuracy_binary']:.3f}, "
          f"AUC={r['auc']:.3f})")
    print("Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):")
    print(r["confusion_binary"])
    print("Per-class detection rate (fraction flagged as anomaly):")
    for c, v in r["per_class_detection"].items():
        marker = "  <- NORMAL (want low)" if c == 0 else ""
        print(f"  F{c}: {v:.3f}{marker}")


Scenario 1  (AnomalyAE, n_f0=858, acc_bin=0.701, AUC=0.949)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[191   9]
 [470 930]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.045  <- NORMAL (want low)
  F1: 0.520
  F2: 0.970
  F3: 0.145
  F4: 1.000
  F5: 0.660
  F6: 0.630
  F7: 0.725

Scenario 2  (AnomalyAE, n_f0=439, acc_bin=0.680, AUC=0.893)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[180  20]
 [492 908]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.100  <- NORMAL (want low)
  F1: 0.475
  F2: 0.915
  F3: 0.185
  F4: 1.000
  F5: 0.710
  F6: 0.575
  F7: 0.680

Scenario 3  (AnomalyAE, n_f0=86, acc_bin=0.409, AUC=0.734)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[185  15]
 [930 470]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.075  <- NORMAL (want low)
  F1: 0.435
  F2: 0.330
  F3: 0.120
  F4: 0.810
  F5: 0.555
  F6: 0.030
  F7: 0.070

Scenario 4  (AnomalyAE, n_f0=41, acc_bin=0.277, AUC=